# K=11 Producao -- Orquestracao (execucao local ou Colab)

Pipeline de producao do modelo de regressao Bayesiana hierarquica K=11
(10 features baseline + mode_bin) para o produto **Diagnostico de Posicionamento**.

## Estrutura desta pasta (tudo num lugar so)

```
scripts/k11_pipeline/
├── 06_k11_pipeline.ipynb     # este notebook
├── train.py                  # treino NUTS K=11
├── evaluate.py               # metricas + asserts
├── export.py                 # exporta JSON para Next.js
└── spotify_tracks_limpo.parquet  # dataset
```

## Pre-requisitos

- **Python 3.10+**
- **GPU NVIDIA recomendada** (T4, RTX 3060+, A100). Sem GPU, o NUTS demora ~30h.
- Drivers CUDA 12.x (se for usar GPU local)
- No Colab: T4 gratuita (sessao de 12h)

## Setup rapido (Colab)

1. Faca upload da pasta `k11_pipeline/` inteira para o Colab (pode ser via zip)
2. Abra `06_k11_pipeline.ipynb` no Colab
3. Runtime > Change runtime type > T4 GPU
4. Run all cells

## Setup rapido (local)

```bash
cd /caminho/para/insights-spotfy-grupo-4/scripts/k11_pipeline
pip install -r ../../../requirements.txt  # ou o caminho equivalente
jupyter lab  # abre este notebook
```

## Arquitetura

- **Target:** `log(popularity + 1)` -> score 0-100 apos `exp() - 1`
- **Features (K=11):** danceability, energy, loudness, speechiness, acousticness, instrumentalness, liveness, valence, tempo, explicit, mode_bin
- **Hierarquica:** 107 generos, intercept + slopes especificos, prior nao-centrado
- **Sampler:** NUTS via NumPyro, 4 chains x 1000 draws, tune=1500, target_accept=0.9
  - GPU (T4/A100): ~3h
  - CPU so: ~30h
- **Validacao:** Train/Val/Test 70/15/15 com SEED=42, asserts RMSE<18, R2>0.30, HDI 0.90-0.97

## Etapas

1. `train.py` -- fit NUTS (~3h em GPU)
2. `evaluate.py` -- metricas em Val e Test, gera `q11_summary.json`
3. `export.py` -- converte NetCDF em JSON para o backend Next.js


In [ ]:
# Detecta GPU NVIDIA ANTES de instalar JAX (evita conflito com a versao CPU ja instalada)
import subprocess
try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader,nounits'],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    has_gpu = True
    print(f'GPU NVIDIA detectada: {out}')
except Exception:
    has_gpu = False
    print('Nenhuma GPU NVIDIA detectada (ou nvidia-smi indisponivel).')

# Instala JAX CORRETAMENTE (cuda12 se GPU, senao CPU) -- --upgrade --force-reinstall
# garante que sobrescreve qualquer jax CPU ja instalado como dependencia do pymc
if has_gpu:
    print('Instalando jax[cuda12]==0.5.3 (GPU) ...')
    !pip install -q --upgrade --force-reinstall "jax[cuda12]==0.5.3"
else:
    print('Instalando jax==0.5.3 (CPU) ...')
    !pip install -q --upgrade --force-reinstall "jax==0.5.3"

# Instala o resto das dependencias
!pip install -q pymc==6.3.1 arviz==1.3.0 pytensor==3.3.0 numpyro==0.21.0 pandas pyarrow scipy scikit-learn

# Verifica backend do JAX
import jax
print()
print('jax version:', jax.__version__)
print('jax devices:', jax.devices())
print('jax backend:', jax.default_backend())

if has_gpu and jax.default_backend() != 'gpu':
    print()
    print('*** ATENCAO: GPU foi detectada mas JAX nao esta usando CUDA. ***')
    print('Va em Runtime > Restart runtime, e rode esta celula (C2) novamente.')
elif has_gpu:
    print()
    print('[OK] JAX em GPU. NUTS levara ~3h em T4.')
else:
    print()
    print('[AVISO] JAX em CPU. NUTS levara ~30h. Considere Colab T4.')


In [ ]:
import os
import sys
from pathlib import Path

# Detecta se estamos no Colab
IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
print(f'Ambiente: {"Google Colab" if IN_COLAB else "Jupyter local"}')
print()

PIPELINE_ROOT = None

if IN_COLAB:
    # Tenta montar Drive (silenciosamente se ja estiver montado)
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive', force_remount=False)
            print('Drive montado em /content/drive')
        else:
            print('Drive ja estava montado em /content/drive')
        print()
    except Exception as e:
        print(f'Drive nao disponivel (continuando sem): {e}')
        print()

    # Procura a pasta k11_pipeline (que contem train.py + parquet)
    candidates = [
        # Upload direto da pasta em /content/
        Path('/content/k11_pipeline'),
        # Se subiu apenas o zip, o usuario pode ter extraido em qualquer lugar
        # Dentro do Drive
        Path('/content/drive/MyDrive/k11_pipeline'),
        Path('/content/drive/MyDrive/insights-spotfy-grupo-4/scripts/k11_pipeline'),
    ]
    
    found = None
    for c in candidates:
        if c.is_dir() and (c / 'train.py').exists() and (c / 'spotify_tracks_limpo.parquet').exists():
            found = c
            break
    
    # Fallback: busca recursiva por train.py com parquet no mesmo dir
    if found is None:
        print('Procurando recursivamente em /content/...')
        for p in Path('/content').rglob('train.py'):
            if (p.parent / 'spotify_tracks_limpo.parquet').exists():
                found = p.parent
                break
    
    if found is None:
        print('!!! Pasta k11_pipeline nao encontrada.')
        print()
        print('Como subir (escolha UM):')
        print()
        print('Opcao A (mais rapida, ~30s): upload direto da pasta')
        print('  1. Arraste a pasta k11_pipeline/ para o painel Files (esquerda)')
        print('  2. Ela sera salva em /content/k11_pipeline/')
        print('  3. Re-rode esta celula')
        print()
        print('Opcao B (persiste entre sessoes): coloque no Google Drive')
        print('  1. Upload da pasta k11_pipeline/ para /content/drive/MyDrive/k11_pipeline/')
        print('  2. Re-rode esta celula')
        print()
        print('Opcao C (zip): faca upload do zip, depois rode:')
        print('  !unzip k11_pipeline.zip -d /content/')
        raise FileNotFoundError('k11_pipeline nao encontrada')
    
    PIPELINE_ROOT = found
    print(f'k11_pipeline encontrada: {PIPELINE_ROOT}')
    print()

else:
    # Jupyter local: usa o diretorio do proprio notebook
    PIPELINE_ROOT = Path(os.getcwd())
    if not (PIPELINE_ROOT / 'train.py').exists():
        # Tenta subir niveis
        for _ in range(5):
            PIPELINE_ROOT = PIPELINE_ROOT.parent
            if (PIPELINE_ROOT / 'train.py').exists():
                break
    print(f'PIPELINE_ROOT: {PIPELINE_ROOT}')
    print()

os.chdir(PIPELINE_ROOT)

# Garante diretorios que os scripts escrevem
(PIPELINE_ROOT / 'artifacts').mkdir(parents=True, exist_ok=True)
(PIPELINE_ROOT / 'relatorio' / 'analises' / 'resultados').mkdir(parents=True, exist_ok=True)

# Verificacoes finais
print('=== Estrutura ===')
for f in ['train.py', 'evaluate.py', 'export.py', 'spotify_tracks_limpo.parquet', 'artifacts/', 'relatorio/analises/resultados/']:
    full = PIPELINE_ROOT / f
    status = '[OK]' if full.exists() else '[FALTA]'
    print(f'  {status}  {f}')

# Assertions
assert (PIPELINE_ROOT / 'train.py').exists(), 'train.py nao encontrado'
assert (PIPELINE_ROOT / 'evaluate.py').exists(), 'evaluate.py nao encontrado'
assert (PIPELINE_ROOT / 'export.py').exists(), 'export.py nao encontrado'
assert (PIPELINE_ROOT / 'spotify_tracks_limpo.parquet').exists(), 'parquet nao encontrado'
print()
print('Tudo certo. Pode prosseguir para C4.')


In [ ]:
!python train.py 2>&1 | tee train.log


In [ ]:
!python evaluate.py 2>&1 | tee evaluate.log


In [ ]:
!python export.py 2>&1 | tee export.log


In [ ]:
import json
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

print('=== Artefatos gerados ===\n')
for f in sorted(Path('artifacts').iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s}  {size_kb:8.1f} KB')

print('\n=== Metricas ===\n')
summary_path = Path('relatorio/analises/resultados/q11_summary.json')
assertions_passed = False
if summary_path.exists():
    with open(summary_path) as fh:
        summary = json.load(fh)
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    # Chave flat (top-level) -- escrita por evaluate.py
    assertions_passed = summary.get('assertions_passed', False)
    print('\n[OK] assertions_passed:', assertions_passed)
else:
    print('AVISO: q11_summary.json nao encontrado -- verifique se evaluate.py rodou sem erro.')

# Secao de download para Colab
if IN_COLAB and assertions_passed:
    print('\n=== Download dos artefatos (Colab) ===\n')
    print('Os artefatos estao em:', Path('artifacts').resolve())
    print()
    print('Para usar no backend Next.js local, copie de volta para o repo:')
    print('  scripts/k11_pipeline/artifacts/*.json  ->  artifacts/  (raiz do repo)')
    print('  scripts/k11_pipeline/artifacts/*.gz    ->  artifacts/  (raiz do repo)')
    print()
    print('Comandos equivalentes no PowerShell local:')
    print('  Copy-Item scripts/k11_pipeline/artifacts/* -Destination artifacts/ -Force')
    print()
    print('Opcao 1 (manual): baixe um por um pelo painel Files (botao direito > Download)')
    print()
    print('Opcao 2 (zip automatico): disparando download abaixo...')
    print()
    # Dispara download do zip
    from google.colab import files
    import shutil
    zip_path = shutil.make_archive('k11_artifacts', 'zip', 'artifacts')
    files.download(zip_path)


## Proximos passos

### Se `assertions_passed == true:`

1. **Copiar artefatos para o repo (para o backend Next.js):**

   O backend Next.js espera os artefatos em `<repo_root>/artifacts/`, nao em `scripts/k11_pipeline/artifacts/`.

   No PowerShell, na raiz do repo:
   ```powershell
   Copy-Item scripts/k11_pipeline/artifacts/* -Destination artifacts/ -Force
   ```

   Ou no bash:
   ```bash
   cp scripts/k11_pipeline/artifacts/* artifacts/
   ```

2. **Commitar (opcional):**
   ```bash
   git add scripts/k11_pipeline/ artifacts/ scripts/k11_pipeline/*.log
   git commit -m "feat: K=11 modelo treinado e validado"
   git push
   ```

3. **Subir o backend Next.js (em outra pasta ou outra maquina):**
   ```bash
   cd /caminho/para/insights-spotfy-grupo-4
   # garantir que os artefatos estao em ./artifacts/
   npm install
   cp .env.local.example .env.local
   # editar .env.local e colocar OPENROUTER_API_KEY=sk-or-v1-...
   npm run dev
   ```

4. **Testar o endpoint:**
   ```bash
   curl -X POST http://localhost:3000/api/diagnose \
     -H "Content-Type: application/json" \
     -d '{
       "track_features": {
         "danceability": 0.7, "energy": 0.5, "loudness": -5.0,
         "speechiness": 0.05, "acousticness": 0.3,
         "instrumentalness": 0.0, "liveness": 0.1,
         "valence": 0.6, "tempo": 120.0, "explicit": 0, "mode_bin": 0
       },
       "genero": "sertanejo"
     }'
   ```

### Se `assertions_passed == false:`

Investigar `q11_summary.json` e ver qual metrica falhou:

| Metrica | Falha comum | Acao |
|---------|-------------|------|
| RMSE >= 18 | Modelo nao captura variancia | Aumentar K? (nao recomendado, Q8 v2 mostrou overfit) |
| R2 <= 0.30 | Pouca variancia explicada | Aceitar -- pode ser teto do problema |
| HDI fora de [0.90, 0.97] | Calibracao ruim | Ajustar priors sigma_alpha/sigma_beta |

## Caveats do modelo

- **Genero deve ser conhecido** -- o dropdown tem 107 opcoes apos filtro nao-musical
- **Score e preditivo, nao causal** -- diz "o que costuma acontecer", nao "como fazer hit"
- **Calibrado em popularity do Spotify (0-100)**, nao em qualidade musical
- **NUTS aproxima o posterior** -- HDI e uma estimativa, nao certeza

## Se algo der errado

Logs ficam salvos em:
- `train.log` -- log completo do treino (inclui R-hat, ESS, divergencias)
- `evaluate.log` -- log da avaliacao
- `export.log` -- log do export

Para debug, rode os scripts diretamente no terminal (cwd = scripts/k11_pipeline/):
```bash
python train.py
```
e veja o erro com traceback completo.
